# Fine-Tuning on Google Colab

LoRA fine-tuning for `meta-llama/Llama-2-7b-hf` (4-bit, rank 16). Requires Colab GPU and Hugging Face access to the gated base model.

1. Upload or clone this repo to `/content/hospital-fiap-assistant`
2. Run `prepare_dataset.py` if `data/processed/` is empty
3. Set `HF_TOKEN` and run training cells below

In [ ]:
# Colab: install dependencies (skip if already installed locally)
!pip install -q torch transformers peft bitsandbytes accelerate datasets python-dotenv

## Hugging Face login

Request access to [Llama 2](https://huggingface.co/meta-llama/Llama-2-7b-hf), then authenticate:

- **Colab secrets:** add `HF_TOKEN` in the sidebar
- **Or interactively:** uncomment `login()` below and paste your token

In [ ]:
import os
from pathlib import Path

# from huggingface_hub import login
# login()  # paste HF token when prompted

ROOT = Path("/content/hospital-fiap-assistant")
if not ROOT.exists():
    ROOT = Path("..").resolve() if (Path("..") / "fine_tuning" / "train.py").exists() else Path(".").resolve()
os.chdir(ROOT)
print("Working directory:", ROOT)

In [ ]:
# Optional: build dataset if missing
train_path = ROOT / "data/processed/train.jsonl"
if not train_path.exists():
    !python fine_tuning/prepare_dataset.py --pubmedqa data/synthetic/pubmedqa_sample.jsonl

In [ ]:
from fine_tuning.train import build_arg_parser, run_training

# Full training: remove --max-steps for 3 epochs on GPU
parser = build_arg_parser()
args = parser.parse_args(["--max-steps", "2"])
metrics = run_training(args)
metrics

In [ ]:
# Adapter saved to artifacts/lora_adapter/ — download and set LORA_ADAPTER_PATH locally
print("skipped_training:", metrics.get("skipped_training"))
print("output:", metrics.get("output_dir", "artifacts/lora_adapter"))
print("epochs:", metrics.get("epochs"))